# VEGAN Demo — Microscopic Medical Image Segmentation

**Paper:** [VEGAN: A Vision-language and Edge-enhanced GAN-based Microscopic Medical Image Segmentation Model](https://doi.org/10.1109/MECO66322.2025.11049114)  
**Published at:** IEEE MECO 2025  
**Author:** Souvik Pramanik (DIAT Pune, DRDO)

This notebook demonstrates how to run inference with VEGAN. No local setup required — just run all cells in order.

> **Runtime:** Make sure you have a GPU runtime enabled. Go to `Runtime > Change runtime type > GPU`.

## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
# Clone the VEGAN repository
!git clone https://github.com/souvikpramanik-medimg/VEGAN.git
%cd VEGAN

# Install dependencies
!pip install -q -r requirements.txt

print('✅ Setup complete!')

## 2. Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image

from vegan.utils.edge_maps import compute_combined_edge_map
from vegan.utils.metrics import dice_score, iou_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 3. Load a Sample Image

In [ ]:
# Upload your own image, or use a sample
# from google.colab import files
# uploaded = files.upload()

# For demo: download a sample MoNuSeg image
!wget -q -O sample.png https://raw.githubusercontent.com/souvikpramanik-medimg/VEGAN/main/docs/assets/sample_monuseg.png || echo 'Add a sample image to docs/assets/ in your repo'

image_path = 'sample.png'
image = np.array(Image.open(image_path).resize((256, 256)))

plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.title('Input Image')
plt.axis('off')
plt.show()

## 4. Compute Combined Edge Map

In [ ]:
edge_map = compute_combined_edge_map(image)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image)
axes[0].set_title('Input Image')
axes[0].axis('off')

axes[1].imshow(edge_map, cmap='gray')
axes[1].set_title('Combined Edge Map\n(Sobel + Canny + Laplacian)')
axes[1].axis('off')

plt.tight_layout()
plt.show()
print(f'Edge map shape: {edge_map.shape}, range: [{edge_map.min():.3f}, {edge_map.max():.3f}]')

## 5. Run Inference

> **Note:** Load your trained checkpoint below. If you don't have a checkpoint yet, the notebook will skip inference and show only the preprocessing pipeline.

In [ ]:
import os

CHECKPOINT_PATH = 'checkpoints/vegan_monuseg.pth'  # Update this path

if os.path.exists(CHECKPOINT_PATH):
    from vegan.models import AttentionUNet
    model = AttentionUNet().to(device)
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    model.eval()

    # Preprocess
    img_tensor = torch.from_numpy(image.transpose(2, 0, 1)).float().unsqueeze(0) / 255.0
    edge_tensor = torch.from_numpy(edge_map).float().unsqueeze(0).unsqueeze(0)
    img_tensor = img_tensor.to(device)
    edge_tensor = edge_tensor.to(device)

    with torch.no_grad():
        pred_mask = torch.sigmoid(model(img_tensor, edge_tensor))

    pred_np = pred_mask.squeeze().cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image)
    axes[0].set_title('Input')
    axes[0].axis('off')
    axes[1].imshow(edge_map, cmap='gray')
    axes[1].set_title('Edge Map')
    axes[1].axis('off')
    axes[2].imshow(pred_np > 0.5, cmap='gray')
    axes[2].set_title('Predicted Mask')
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f'No checkpoint found at {CHECKPOINT_PATH}.')
    print('Preprocessing pipeline (edge maps) demonstrated above.')
    print('To run full inference, add your trained .pth checkpoint.')

## Citation

If you use this work, please cite:

```bibtex
@inproceedings{maity2025vegan,
  title     = {{VEGAN}: A Vision-language and Edge-enhanced {GAN}-based Microscopic Medical Image Segmentation Model},
  author    = {Maity, Gouranga and Pramanik, Souvik and Mandal, Diptarka and Kaplun, Dmitrii and Gulvanskii, Vyacheslav and Sarkar, Ram},
  booktitle = {Proceedings of the 14th Mediterranean Conference on Embedded Computing (MECO)},
  year      = {2025},
  publisher = {IEEE},
  doi       = {10.1109/MECO66322.2025.11049114}
}
```